# Rare-event simulation for a simple Queue-Reactive limit order book model

We study two independent queues:

- Ask queue: $Q_{\mathrm{ask}}(t)$
- Bid queue: $Q_{\mathrm{bid}}(t)$

Each queue is a continuous-time birth-death process:

$$
Q_{\mathrm{ask}} \to Q_{\mathrm{ask}}+1
\quad \text{at rate } \lambda_{\mathrm{ask}}^+
$$

$$
Q_{\mathrm{ask}} \to Q_{\mathrm{ask}}-1
\quad \text{at rate } \lambda_{\mathrm{ask}}^-
$$

and similarly for the bid queue.

The stopping times are

$$
\tau_{\mathrm{ask}}
=
\inf \{t \geq 0 : Q_{\mathrm{ask}}(t)=0\},
$$

$$
\tau_{\mathrm{bid}}
=
\inf \{t \geq 0 : Q_{\mathrm{bid}}(t)=0\}.
$$

The rare event of interest is

$$
\mathbb{P}(\tau_{\mathrm{ask}} < \tau_{\mathrm{bid}}),
$$

with initial condition

$$
Q_{\mathrm{ask}}(0)=q_{\mathrm{ask},0} \geq 80,
\qquad
Q_{\mathrm{bid}}(0)=q_{\mathrm{bid},0} \leq 4.
$$

The event is rare because the ask queue starts large while the bid queue starts small.

In [31]:
import pandas as pd

from helpers.helpers_EvRareSIMPLE import *


# Model parameters

The default parameters below are chosen to make the event

$$
\tau_{\mathrm{ask}} < \tau_{\mathrm{bid}}
$$

rare when $q_{\mathrm{ask},0}=80$ and $q_{\mathrm{bid},0}=4$.

You should modify these values according to the parameters used in your project.

In [32]:
# Initial queue sizes
q_ask0 = 80
q_bid0 = 4

# Original Poisson intensities
lambda_ask_plus = 1.2
lambda_ask_minus = 1.5

lambda_bid_plus = 1.2
lambda_bid_minus = 1.5

# Random seed
seed = 12345


# Utility functions

For a Monte Carlo estimator

$$
\widehat p = \frac{1}{n}\sum_{i=1}^n X_i,
$$

the standard error is

$$
\mathrm{SE}(\widehat p)
=
\frac{\widehat{\sigma}}{\sqrt n}.
$$

The relative error is

$$
\frac{\mathrm{SE}(\widehat p)}{\widehat p}.
$$

For rare events, the relative error is often more informative than the absolute standard error.

# Naive Monte Carlo

We simulate the exact continuous-time Markov chain.

At each step, the next event is one of:

1. Ask addition,
2. Ask removal,
3. Bid addition,
4. Bid removal.

The total intensity is

$$
\Lambda
=
\lambda_{\mathrm{ask}}^+
+
\lambda_{\mathrm{ask}}^-
+
\lambda_{\mathrm{bid}}^+
+
\lambda_{\mathrm{bid}}^-.
$$

The waiting time until the next event is exponential:

$$
\Delta t \sim \mathrm{Exp}(\Lambda).
$$

Conditional on an event happening, the event type is chosen with probability proportional to its intensity.

The simulation stops when either queue reaches zero.

In [33]:
rates_original = {
    "lambda_ask_plus": lambda_ask_plus,
    "lambda_ask_minus": lambda_ask_minus,
    "lambda_bid_plus": lambda_bid_plus,
    "lambda_bid_minus": lambda_bid_minus,
}

# Retained baseline: exactly 10^5 independent trajectories.
n_naive = 10**5

naive_summary, naive_raw = naive_monte_carlo(
    n_samples=n_naive,
    q_ask0=q_ask0,
    q_bid0=q_bid0,
    rates=rates_original,
    seed=seed,
)

pd.DataFrame([naive_summary])


,method,n,estimate,sample_variance,standard_error,relative_error,ci95_low,ci95_high,rare_events_observed,mean_tau,mean_n_events
0,Naive Monte Carlo,100000,0.0007,0.0007,0.000084,0.119482,0.000536,0.000864,70,13.213147,71.33563


# Naive Monte Carlo diagnostic

For rare events, the expected number of rare events observed is approximately

$$
np.
$$

With the retained baseline of $n=10^5$ samples, a probability around $10^{-4}$ yields about $10$ observed rare events on average. This is enough for a simple reference estimate, but its relative error remains substantial.


In [34]:
print("Naive Monte Carlo diagnostics")
print("-----------------------------")
print(f"Estimate:              {naive_summary['estimate']:.6e}")
print(f"95% CI:                [{naive_summary['ci95_low']:.6e}, {naive_summary['ci95_high']:.6e}]")
print(f"Rare events observed:  {naive_summary['rare_events_observed']}")
print(f"Relative error:        {naive_summary['relative_error']:.4f}")
print(f"Mean tau:              {naive_summary['mean_tau']:.4f}")
print(f"Mean number of events: {naive_summary['mean_n_events']:.2f}")

Naive Monte Carlo diagnostics
-----------------------------
Estimate:              7.000000e-04
95% CI:                [5.360712e-04, 8.639288e-04]
Rare events observed:  70
Relative error:        0.1195
Mean tau:              13.2131
Mean number of events: 71.34


# Fixed Multilevel Splitting

Fixed splitting keeps the original probability law, but decomposes the rare event into less rare conditional events.

Define decreasing ask levels:

$$
q_{\mathrm{ask},0}
>
\ell_1
>
\ell_2
>
\cdots
>
\ell_m
=
0.
$$

Then

$$
\mathbb{P}(\tau_{\mathrm{ask}} < \tau_{\mathrm{bid}})
=
\prod_{k=1}^m
\mathbb{P}
\left(
\text{reach level } \ell_k \text{ before bid hits 0}
\mid
\text{reached level } \ell_{k-1}
\right).
$$

At each level:

1. simulate particles until they either reach the next ask level or the bid queue hits zero;
2. keep only successful particles;
3. resample successful particles;
4. continue to the next level.


# Running fixed splitting once

The choice of levels is important.

Here we use ask levels decreasing by 10:

$$
80,70,60,\ldots,10,0.
$$

If some conditional probabilities are too small, use more intermediate levels.  
If they are too close to 1, use fewer levels.

In [35]:
levels = list(range(q_ask0, 0, -10)) + [0]
levels

[80, 70, 60, 50, 40, 30, 20, 10, 0]

In [36]:
splitting_out = fixed_multilevel_splitting(
    n_particles=2_000,
    levels=levels,
    q_ask0=q_ask0,
    q_bid0=q_bid0,
    rates=rates_original,
    seed=seed + 2000,
)

splitting_out

{'method': 'Fixed Multilevel Splitting',
 'estimate': 0.000644829319391082,
 'n_particles': 2000,
 'levels': [80, 70, 60, 50, 40, 30, 20, 10, 0],
 'conditional_probabilities': [0.1925,
  0.296,
  0.385,
  0.4365,
  0.486,
  0.5145,
  0.526,
  0.512],
 'survivors_by_level': [385, 592, 770, 873, 972, 1029, 1052, 1024],
 'mean_ask_by_level': [np.float64(70.0),
  np.float64(60.0),
  np.float64(50.0),
  np.float64(40.0),
  np.float64(30.0),
  np.float64(20.0),
  np.float64(10.0),
  np.float64(0.0)],
 'mean_bid_by_level': [np.float64(6.2727272727272725),
  np.float64(8.20777027027027),
  np.float64(9.585714285714285),
  np.float64(10.672394043528064),
  np.float64(11.411522633744855),
  np.float64(11.881438289601554),
  np.float64(11.959125475285171),
  np.float64(12.4736328125)],
 'total_events': 1348515,
 'failed': False}

In [37]:
print("Fixed Multilevel Splitting")
print("--------------------------")
print(f"Estimate: {splitting_out['estimate']:.6e}")
print(f"Failed:   {splitting_out['failed']}")
print("Conditional probabilities:")

for k, p_cond in enumerate(splitting_out["conditional_probabilities"], start=1):
    print(f"  Level {levels[k-1]} -> {levels[k]}: {p_cond:.6f}")

Fixed Multilevel Splitting
--------------------------
Estimate: 6.448293e-04
Failed:   False
Conditional probabilities:
  Level 80 -> 70: 0.192500
  Level 70 -> 60: 0.296000
  Level 60 -> 50: 0.385000
  Level 50 -> 40: 0.436500
  Level 40 -> 30: 0.486000
  Level 30 -> 20: 0.514500
  Level 20 -> 10: 0.526000
  Level 10 -> 0: 0.512000


# Repeating splitting to estimate its variability

One run of splitting gives one estimator.

To estimate a standard error, we repeat the whole splitting procedure several times independently.

In [38]:
splitting_summary, splitting_repeats = repeat_fixed_splitting(
    n_repeats=20,
    n_particles=1_000,
    levels=levels,
    q_ask0=q_ask0,
    q_bid0=q_bid0,
    rates=rates_original,
    seed=seed + 3000,
)

pd.DataFrame([splitting_summary])

,method,n,estimate,sample_variance,standard_error,relative_error,ci95_low,ci95_high,n_repeats,n_particles,failed_repeats,mean_total_events
0,Fixed Multilevel Splitting,20,0.000722,2.624470e-08,0.000036,0.050197,0.000651,0.000793,20,1000,0,693974.15
